# Fresh Unified 1 — Run 5 (answer-disjoint corrective run)

This notebook is pinned to the merged Fresh AI evaluation fix. It does **not** modify or replace Run 3 or Run 4.

## Required Kaggle setup
1. Set **Accelerator = GPU** (one T4/P100-class GPU is sufficient for this configuration).
2. Set **Internet = On**.
3. Under **Add Input → Your Work → Notebook Output Files**, attach the output of `memphisarafat/fresh-unified-training-kaggle` (Run 3). It must contain:
   - `fresh-unified-1-run3.pt`
   - `tokenizer-v2.json`
4. Run cells from top to bottom. A cell stops on any missing artifact, hash mismatch, or leakage.

## Run 5 controls
- Initializes from the verified Run 3 checkpoint only.
- Reuses the exact Run 3 tokenizer; no new tokenizer is trained.
- Rebuilds the Fresh docs split by normalized answer content.
- Builds the authorized SFT mix and audits held-out recall answers against both train and validation.
- Runs a short initialization smoke test before the 654-step controlled training run.
- Evaluates Run 3 and Run 5 on identical recall and validation inputs.
- Keeps the candidate experimental. No registry change, runtime switch, or production promotion is performed.

**Do not train if the leakage audit or either Run 3 SHA-256 check fails.**

In [ ]:
# 1. Runtime, repository pin, and dependencies
import os, sys, json, hashlib, shutil, subprocess, zipfile
from pathlib import Path

W = Path("/kaggle/working")
REPO = W / "fresh-web-lite"
PINNED_COMMIT = "73da0be7d572e56bb92b23cc869be68d31ed97e3"
RUN3_CHECKPOINT_SHA256 = "298b558e89edab77745834389cb2b1979e632979f9be7b4adad1892c2f2fc25d"
RUN3_TOKENIZER_SHA256 = "dc820bac32dd5f3bc0b47eac545d4baa33cde93da05014b2d90d0903b0c7b06c"

def run(cmd, log_path=None, check=True):
    print("\n$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=str(REPO) if REPO.exists() else None,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    log = open(log_path, "w", encoding="utf-8") if log_path else None
    output = []
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
            output.append(line)
            if log:
                log.write(line)
    finally:
        if log:
            log.close()
    code = proc.wait()
    if check and code:
        raise RuntimeError(f"Command failed ({code}): {cmd}")
    return code, "".join(output)

rc, gpu_info = run("nvidia-smi -L", check=False)
if rc != 0:
    raise RuntimeError("GPU not available. In Kaggle settings choose Accelerator = GPU before training.")

if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/kanarafat54-boop/fresh-web-lite.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", PINNED_COMMIT], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", PINNED_COMMIT], check=True)
os.chdir(REPO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tokenizers"], check=True)
import torch
print("Pinned commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


In [ ]:
# 2. Regenerate and verify the answer-disjoint Fresh docs data
run("node scripts/generate-fresh-knowledge-sft.mjs")
run("npm run test:fresh-knowledge-sft")
print("Fresh docs train/evaluation generator and regression tests: PASS")


In [ ]:
# 3. Build the exact Run 5 mix and audit final train + validation splits
D = W / "fresh-run5-data"
MIX = D / "mix"
D.mkdir(parents=True, exist_ok=True)

run(f"python3 training/build_training_mix.py --recipe training/mix_recipe_sft_v1.json --out-dir {MIX}")
AUDIT = D / "recall-leakage-audit.json"
run(
    f"python3 training/audit_fresh_eval_leakage.py "
    f"--train {MIX / 'train.jsonl.gz'} "
    f"--validation {MIX / 'validation.jsonl.gz'} "
    f"--recall-eval data/fresh-eval/fresh-knowledge-recall-v1.jsonl "
    f"--out {AUDIT}"
)
audit = json.loads(AUDIT.read_text())
assert audit["passed"] is True, "STOP: final-corpus leakage audit failed"
assert audit["answerOverlapCount"] == 0
assert audit["validationAnswerOverlapCount"] == 0
assert audit["questionOverlapCount"] == 0
print("LEAKAGE GATE: PASS — zero answer overlap with train or validation.")
print("Train rows:", audit["trainRows"], "| Recall prompts:", audit["evaluationPrompts"])


In [ ]:
# 4. Find and verify the exact Run 3 checkpoint and tokenizer
def unique_kaggle_input(filename):
    matches = list(Path("/kaggle/input").rglob(filename))
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one {filename} under /kaggle/input; found {len(matches)}. "
            "Attach the Run 3 notebook output and remove duplicate inputs."
        )
    return matches[0]

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

RUN3_CHECKPOINT = unique_kaggle_input("fresh-unified-1-run3.pt")
RUN3_TOKENIZER = unique_kaggle_input("tokenizer-v2.json")
checkpoint_hash = sha256_file(RUN3_CHECKPOINT)
tokenizer_hash = sha256_file(RUN3_TOKENIZER)
print("Run 3 checkpoint:", RUN3_CHECKPOINT, checkpoint_hash)
print("Run 3 tokenizer:", RUN3_TOKENIZER, tokenizer_hash)
if checkpoint_hash != RUN3_CHECKPOINT_SHA256:
    raise RuntimeError("STOP: Run 3 checkpoint SHA-256 mismatch.")
if tokenizer_hash != RUN3_TOKENIZER_SHA256:
    raise RuntimeError("STOP: Run 3 tokenizer SHA-256 mismatch.")
print("Run 3 artifact identity gate: PASS")


In [ ]:
# 5. Encode Run 5 data with the verified Run 3 tokenizer (do not train a new tokenizer)
for split in ("train", "validation"):
    run(
        f"python3 training/fresh_tokenizer_v2.py encode "
        f"--tokenizer {RUN3_TOKENIZER} "
        f"--input {MIX / (split + '.jsonl.gz')} "
        f"--output {D / (split + '.bin')}"
    )
print("Encoded streams:")
for p in (D / "train.bin", D / "validation.bin"):
    print(p, f"{p.stat().st_size:,} bytes")


In [ ]:
# 6. Short full-architecture initialization smoke test from Run 3
SMOKE_DIR = W / "fresh-run5-smoke"
if SMOKE_DIR.exists():
    shutil.rmtree(SMOKE_DIR)

smoke_cmd = (
    f"python3 training/train_fresh_unified_full.py "
    f"--train-bin {D / 'train.bin'} --val-bin {D / 'validation.bin'} "
    f"--tokenizer {RUN3_TOKENIZER} --spec training/model_spec.json "
    f"--init-from {RUN3_CHECKPOINT} "
    f"--seq-len 256 --micro-batch 1 --accum 1 --max-steps 2 "
    f"--lr 0.0001 --warmup 1 --eval-every 1 --save-every 2 --eval-blocks 2 "
    f"--seed 42 --out-dir {SMOKE_DIR} --run-id fresh-run5-init-smoke"
)
run(smoke_cmd, log_path=D / "run5-smoke.log")
smoke_manifest = json.loads((SMOKE_DIR / "fresh-run5-init-smoke.manifest.json").read_text())
assert smoke_manifest["executionStatus"] == "completed"
assert smoke_manifest["pipelineSmokeTest"] is True
assert smoke_manifest["initFromSha256"] == RUN3_CHECKPOINT_SHA256
print("Run 5 initialization smoke test: PASS")


In [ ]:
# 7. Controlled Run 5 training — 654 steps, initialized from Run 3
RUN5_DIR = W / "train-run5"
RUN5_MANIFEST = RUN5_DIR / "fresh-unified-1-run5.manifest.json"
RUN5_CHECKPOINT = RUN5_DIR / "fresh-unified-1-run5.pt"

if RUN5_MANIFEST.exists() and RUN5_CHECKPOINT.exists():
    saved = json.loads(RUN5_MANIFEST.read_text())
    if sha256_file(RUN5_CHECKPOINT) != saved["checkpointSha256"]:
        raise RuntimeError("Existing Run 5 checkpoint does not match its manifest; preserve it and inspect before continuing.")
    print("A completed Run 5 artifact already exists; not overwriting it.")
else:
    train_cmd = (
        f"python3 training/train_fresh_unified_full.py "
        f"--train-bin {D / 'train.bin'} --val-bin {D / 'validation.bin'} "
        f"--tokenizer {RUN3_TOKENIZER} --spec training/model_spec.json "
        f"--init-from {RUN3_CHECKPOINT} --mix-manifest {MIX / 'manifest.json'} "
        f"--seq-len 1024 --micro-batch 8 --accum 4 --max-steps 654 "
        f"--lr 0.0001 --warmup 20 --eval-every 100 --save-every 50 --seed 42 "
        f"--out-dir {RUN5_DIR} --run-id fresh-unified-1-run5"
    )
    run(train_cmd, log_path=D / "run5-training.log")

run5_manifest = json.loads(RUN5_MANIFEST.read_text())
assert run5_manifest["executionStatus"] == "completed"
assert run5_manifest["steps"] == 654
assert run5_manifest["pipelineSmokeTest"] is False
assert run5_manifest["initFromSha256"] == RUN3_CHECKPOINT_SHA256
assert run5_manifest["tokenizerSha256"] == RUN3_TOKENIZER_SHA256
assert run5_manifest["productionReady"] is False
assert sha256_file(RUN5_CHECKPOINT) == run5_manifest["checkpointSha256"]
print("Run 5 completed:", run5_manifest["checkpointSha256"])
print("Held-out loss:", run5_manifest["heldOutLoss"], "| perplexity:", run5_manifest["heldOutPerplexity"])


In [ ]:
# 8. Paired evaluation: Run 3 baseline and Run 5 candidate on identical inputs
EVAL3 = D / "eval-run3.json"
EVAL5 = D / "eval-run5.json"
common = (
    f"--tokenizer {RUN3_TOKENIZER} "
    f"--recall-eval data/fresh-eval/fresh-knowledge-recall-v1.jsonl "
    f"--validation {MIX / 'validation.jsonl.gz'} "
    f"--limit-recall 428 --limit-per-group 100 --max-new-tokens 120"
)

eval_provenance_path = D / "eval-provenance.json"
expected_eval_provenance = {
    "run3CheckpointSha256": RUN3_CHECKPOINT_SHA256,
    "run5CheckpointSha256": run5_manifest["checkpointSha256"],
    "recallEvalSha256": sha256_file(Path("data/fresh-eval/fresh-knowledge-recall-v1.jsonl")),
    "validationSha256": sha256_file(MIX / "validation.jsonl.gz"),
}
if EVAL3.exists() or EVAL5.exists():
    prior = json.loads(eval_provenance_path.read_text()) if eval_provenance_path.exists() else None
    if prior != expected_eval_provenance:
        EVAL3.unlink(missing_ok=True)
        EVAL5.unlink(missing_ok=True)
eval_provenance_path.write_text(json.dumps(expected_eval_provenance, indent=2) + "\n")

if not EVAL3.exists():
    run(
        f"python3 training/eval_fresh_model.py --checkpoint {RUN3_CHECKPOINT} "
        f"{common} --out {EVAL3} --label run3-baseline",
        log_path=D / "eval-run3.log"
    )
if not EVAL5.exists():
    run(
        f"python3 training/eval_fresh_model.py --checkpoint {RUN5_CHECKPOINT} "
        f"{common} --out {EVAL5} --label run5-candidate",
        log_path=D / "eval-run5.log"
    )

report3 = json.loads(EVAL3.read_text())
report5 = json.loads(EVAL5.read_text())
assert report3["checkpointSha256"] == RUN3_CHECKPOINT_SHA256
assert report5["checkpointSha256"] == run5_manifest["checkpointSha256"]
assert report3["recall"]["questions"] == report5["recall"]["questions"] == 428
assert report3["recallEvalFile"] == report5["recallEvalFile"]
assert report3["perplexityByGroup"].keys() == report5["perplexityByGroup"].keys()
print("Run 3 baseline recall:", report3["recall"])
print("Run 5 candidate recall:", report5["recall"])
print("Run 3 perplexity:", {k: v["perplexity"] for k, v in report3["perplexityByGroup"].items()})
print("Run 5 perplexity:", {k: v["perplexity"] for k, v in report5["perplexityByGroup"].items()})
print("Reminder: these reports do not test safety, truthfulness, abstention, instruction following, or tool use.")


In [ ]:
# 9. Package evidence for review — no production promotion
OUT = W / "fresh-run5-evidence"
OUT.mkdir(parents=True, exist_ok=True)

for src in [
    RUN5_CHECKPOINT,
    RUN5_MANIFEST,
    RUN5_DIR / "fresh-unified-1-run5.samples.txt",
    MIX / "manifest.json",
    AUDIT,
    EVAL3,
    EVAL5,
    eval_provenance_path,
]:
    shutil.copy2(src, OUT / src.name)

shutil.copy2(RUN3_TOKENIZER, OUT / "run3-tokenizer-v2.json")
(OUT / "run3-checkpoint.sha256").write_text(RUN3_CHECKPOINT_SHA256 + "  fresh-unified-1-run3.pt\n")
(OUT / "run3-tokenizer.sha256").write_text(RUN3_TOKENIZER_SHA256 + "  tokenizer-v2.json\n")
(OUT / "run5-run-config.json").write_text(json.dumps({
    "runId": "fresh-unified-1-run5",
    "initFrom": "fresh-unified-1-run3.pt",
    "initFromSha256": RUN3_CHECKPOINT_SHA256,
    "tokenizer": "tokenizer-v2.json",
    "tokenizerSha256": RUN3_TOKENIZER_SHA256,
    "trainMixSha256": run5_manifest.get("trainBinSha256"),
    "validationMixSha256": run5_manifest.get("valBinSha256"),
    "mixManifestSha256": run5_manifest.get("mixManifestSha256"),
    "sequenceLength": 1024,
    "microBatch": 8,
    "gradientAccumulation": 4,
    "steps": 654,
    "learningRate": 0.0001,
    "warmupSteps": 20,
    "seed": 42,
    "evaluationEvery": 100,
    "saveEvery": 50,
    "compile": False,
    "productionReady": False
}, indent=2) + "\n")

zip_path = W / "fresh-unified-1-run5-evidence.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUT.iterdir():
        z.write(p, arcname=p.name)

assert run5_manifest["productionReady"] is False
print("Evidence folder:", OUT)
print("Evidence archive:", zip_path, f"({zip_path.stat().st_size:,} bytes)")
print("Run 5 remains experimental. Do not register or serve this checkpoint yet.")
